# Computing Tempered Energy 

We aim to implement a parallel tempering scheme for annealed MCMC as per Du et al.'s [2024 paper](https://arxiv.org/pdf/2302.11552) on compositional generation with energy-based diffusion. In order to implement parallel tempering into the sampling regime of annealed MCMC, we aim to evaluate the difference between true tempered energy and our approximations.

In [56]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

In [57]:
training_samples_size = 1000
training_samples = torch.normal( 0.5,  0.25, (training_samples_size, 1))

Training process follows standard energy based model training as per [Yu et. al 2020](https://arxiv.org/pdf/1903.08689)

In [58]:
class MLP(nn.Module):
    def __init__(self, x_dim, t_dim, hidden_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(x_dim + t_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, x, t):
        xt = torch.cat([x, t], dim=-1)
        return self.net(xt)

In [67]:
n_epochs = 10
n_timesteps = 1000
batch_size = 32

model = MLP(x_dim=1, t_dim=1, hidden_dim=64, output_dim=1)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

sigma_min = 1e-3
sigma_max = 1.0

for epoch in range(n_epochs):
    idx = torch.randint(0, training_samples.size(0), (batch_size,))
    x_0 = training_samples[idx]

    t = torch.randint(0, n_timesteps, (batch_size, 1), dtype=x_0.dtype)
    sigma = sigma_min * (sigma_max / sigma_min) ** (t / n_timesteps)
    noise = torch.randn_like(x_0)

    x_t = torch.sqrt(1 - sigma**2) * x_0 + sigma * noise
    x_t.requires_grad_(True)

    energy = model(x_t, t)
    pred_score = torch.autograd.grad(energy.sum(), x_t, create_graph=True)[0]

    loss = torch.mean((noise + sigma* pred_score) ** 2)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(loss.item())

0.920454740524292
0.8514986038208008
0.7774123549461365
1.3386114835739136
1.646361231803894
0.8235746622085571
1.084356665611267
1.2347794771194458
1.1763513088226318
0.7986154556274414
